<font size=5>Word2Vec Model Example with Classification Task</font>

Word2Vec is a popular natural language processing (NLP) technique developed by researchers at Google that converts words into numerical vector representations. Unlike traditional methods that treat words as independent symbols, Word2Vec learns the relationships between words by analyzing their context within large text corpora. It uses neural network architectures such as Continuous Bag of Words (CBOW) and Skip-Gram to generate dense word embeddings, where semantically similar words are positioned closer together in a multidimensional vector space. For example, words like “king” and “queen” or “doctor” and “physician” tend to have similar vector representations. Word2Vec is widely used in applications such as text classification, sentiment analysis, machine translation, information retrieval, and as a foundational technique for many modern NLP models.


This is a Word2Vec exaple with downstream classifier task. First, model learns word embeddings from large amount of text data. Then, downstream task of movie review classification is performed on NLTK Movie Review dataset

In [ ]:
# if not already installed in your environment install gensim librarty to access gensim model Word2Vec
#!pip install gensim

In [ ]:
 # ===========================================
# Word2Vec Skip-Gram + Downstream Classifier
# ===========================================
 
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec # for learning word embeddings from large amounts of text data
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import re

In [ ]:
nltk.download('punkt')  # sentence tokenizer from Natural Language Tool Kit
nltk.download('punkt_tab') # a tokenizer model used for sentence boundary detection

In [ ]:
# -------------------------------------------
# Load corpora
# -------------------------------------------

with open("movie_reviews_text.txt", "r", encoding="utf-8") as f:
    movie_reviews_text = f.read().lower()

# Tokenize the text
movie_reviews_tokens = word_tokenize(movie_reviews_text)


In [ ]:
# -------------------------------------------
# Train Word2Vec Models (Skip-Gram)
# -------------------------------------------

print("Training Movie reviews Word2Vec...")
model = Word2Vec(
    sentences=[movie_reviews_tokens],
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,             # sg = 1 → skip-gram
    epochs=20
)


In [ ]:
# -------------------------------------------
# Downstream Task: Movie Review Classification
# -------------------------------------------

# load NLTK movie reviews dataset
from nltk.corpus import movie_reviews
nltk.download('movie_reviews')

documents = []
labels = []

for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        documents.append(movie_reviews.raw(fileid))
        labels.append(1 if category == "pos" else 0)

# Preprocess reviews
def clean(text):
    text = text.lower()
    text = re.sub("[^a-z ]+", " ", text)
    return text

clean_docs = [clean(doc) for doc in documents]
tokenized_docs = [doc.split() for doc in clean_docs]


In [ ]:
# -------------------------------------------
# Build Review Embeddings by Averaging Word Vectors
# -------------------------------------------

def doc_embedding(doc_tokens, model):
    embeddings = []
    for tok in doc_tokens:
        if tok in model.wv:
            embeddings.append(model.wv[tok])
    if len(embeddings) == 0:
        return np.zeros(model.vector_size)
    return np.mean(embeddings, axis=0)

X_movie_reviews = np.array([doc_embedding(doc, model) for doc in tokenized_docs])
y = np.array(labels)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_movie_reviews, y, test_size=0.2, random_state=42)


In [ ]:
# -------------------------------------------
# Train Logistic Regression Classifier
# -------------------------------------------

clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# -------------------------------------------
# Evaluate Model
# -------------------------------------------

pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)


In [ ]:

print("\n========== RESULTS ==========")
print(f"Accuracy using moview reviews embeddings: {acc:.4f}")

print("\n--- Classification Report: Movie Reviews Embeddings ---")
print(classification_report(y_test, pred))
